# IAAIS Chapter 3 — Knowledge Base

The Knowledge Base is the memory and reasoning layer of IAAIS. It stores grounded facts, applies production rules, answers queries, and preserves explanations that a trainer or client can inspect.

## Design decisions

- **Formalism:** a small Horn-style production-rule system. It is expressive enough for relationships among sessions, segments, exercises, repetitions, and support conditions while remaining tractable and inspectable.
- **World assumption:** open world. Missing evidence is `UNKNOWN`, not false. An explicit negative fact is required to report `CONTRADICTED`.
- **Uncertainty:** every fact carries confidence, status, source, and provenance. Proposed evidence can support a candidate but does not silently become confirmed. Derived confidence is bounded by the least confident supporting fact.
- **Explanations:** query results retain fact IDs, rule names, and an indented support tree. This is intended for human review rather than opaque Boolean answers.

In [ ]:
from iaais.knowledge_base import (
    FactStatus,
    KnowledgeBase,
P,
    Polarity,
    Rule,
    V,
)

## Facts and a production rule

This rule says that a sensor segment can become a proposed logging candidate when the session profile, observation, exercise interpretation, and sensor capability are all present.

In [ ]:
kb = KnowledgeBase([
    Rule(
        name="supported_logging_candidate",
        premises=(
            P("session_profile", V("session"), V("profile")),
            P("activity_observation", V("session"), V("segment")),
            P("exercise_segment", V("segment"), V("exercise")),
            P("supported_exercise", V("exercise"), V("profile")),
        ),
        conclusion=P("candidate_log", V("session"), V("segment"), V("exercise")),
    )
])

kb.assert_fact("session_profile", ("S1", "gyro-v1"), fact_id="profile-1", source="session")
kb.assert_fact("activity_observation", ("S1", "SEG1"), fact_id="activity-1", confidence=0.91, source="activity-model")
kb.assert_fact("exercise_segment", ("SEG1", "back_squat"), fact_id="segment-1", confidence=0.88, status=FactStatus.PROPOSED, source="exercise-model")
kb.assert_fact("supported_exercise", ("back_squat", "gyro-v1"), fact_id="catalog-1", source="exercise-catalog")

In [ ]:
candidate = kb.query(P("candidate_log", "S1", "SEG1", "back_squat"))
print("status:", candidate.status.value)
print("derived confidence:", candidate.matches[0].confidence)
print(candidate.explanation_text)

## Open-world and explicit contradiction

The absence of a capability fact is not treated as proof that the sensor cannot support an exercise. A negative fact is required before a positive query can be marked contradicted.

In [ ]:
unknown = kb.query(P("supported_exercise", "deadlift", "gyro-v1"))
print("before explicit evidence:", unknown.status.value)

kb.assert_fact(
    "supported_exercise",
    ("deadlift", "gyro-v1"),
    fact_id="not-supported-1",
    polarity=Polarity.NEGATIVE,
    source="sensor-capability-catalog",
)
contradicted = kb.query(P("supported_exercise", "deadlift", "gyro-v1"))
print("after explicit negative evidence:", contradicted.status.value)

## Knowledge Base → Search Engine integration

The adapter converts Knowledge Base transition facts into the Search Engine's `SearchAction` objects. The Search Engine still owns path exploration; the Knowledge Base owns the evidence and can reject or penalize candidate actions.

In [ ]:
from iaais.knowledge_base import ConstraintCheck, ConstraintDisposition, KnowledgeBaseSearchAdapter, TruthStatus
from iaais.search_engine import SearchAction, SearchAlgorithm, SearchEngine

kb.assert_fact("transition", ("start", "middle"), fact_id="t-1", confidence=0.95)
kb.assert_fact("transition", ("middle", "goal"), fact_id="t-2", confidence=0.90)
kb.assert_fact("transition", ("start", "goal"), fact_id="t-3", confidence=0.99)
kb.assert_fact("blocked_transition", ("start", "goal"), fact_id="blocked-1")

def action_provider(state):
    for fact in kb.query(P("transition", state, V("next"))).matches:
        yield SearchAction(fact.fact_id, fact.arguments[1], metadata={"confidence": fact.confidence})

def constraint_provider(state, action):
    blocked = kb.query(P("blocked_transition", state, action.next_state), infer=False)
    if blocked.status is TruthStatus.ENTAILED:
        return ConstraintCheck(ConstraintDisposition.REJECT, reason="explicitly blocked transition")
    return ConstraintCheck()

adapter = KnowledgeBaseSearchAdapter(kb, action_provider, constraint_provider=constraint_provider)
problem = adapter.make_problem("start", lambda state: state == "goal", "goal")
search_result = SearchEngine(SearchAlgorithm.UNIFORM_COST).search(problem)
print("path:", [step.action for step in search_result.path])
print("states:", search_result.states)
adapter.record_search_result(search_result.path, search_result.status, ["unknown-interval"], run_id="search-demo")

This module now provides the memory and inference boundary described in Section 6. It does not yet model every exercise fact or replace future Classifier, Expert, or Planner modules. Those modules should read and write through this Knowledge Base rather than maintaining separate authoritative copies of domain facts.